# Example 1: Virtual Room with Feedback Delay Network (FDN)

This notebook demonstrates how to create a virtual room in PyRES using the `FDN` class.

In PyRES, the **virtual room** is a digital signal processing (DSP) block that filters the microphone signals to produce loudspeaker signals. Here we use a static Feedback Delay Network (FDN) as the room model. The FDN has:
- Input connections (from system microphones)
- Output connections (to system loudspeakers)
- A specified order (number of delay lines)
- Frequency‑dependent reverberation times ($T_{60}$ at DC and Nyquist)

The DSP operates in the frequency domain, but we can feed it time‑domain impulses and transform back to see the resulting impulse responses.

Let’s start by setting up the environment and importing the necessary modules.

## 1. Imports and Path Configuration

We need to add the parent directory of the notebook to Python’s path so that PyRES can be imported. This assumes the notebook is located in the `examples/` folder of the repository.

In [ ]:
import sys
import os
# Add parent directory to path (to find PyRES)
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import matplotlib.pyplot as plt

# FLAMO (used for frequency‑domain processing)
from flamo import dsp
from flamo.functional import signal_gallery

# PyRES modules
from PyRES.virtual_room import FDN
from PyRES.plots import plot_irs_compare, plot_spectrograms_compare

## 2. Time–Frequency Parameters

These parameters control the FFT size, sampling rate, and anti‑aliasing decay.  
- `samplerate`: sampling frequency in Hz  
- `nfft`: number of FFT points (here set to 3 seconds)  
- `alias_decay_db`: attenuation applied to avoid time‑domain aliasing (0 = no extra decay)

In [ ]:
samplerate = 48000          # Hz
nfft = samplerate * 3       # FFT size (3 seconds)
alias_decay_db = 0          # No anti‑aliasing decay

## 3. Virtual Room Configuration

We choose the number of microphones (inputs) and loudspeakers (outputs) for the system. Then we create an instance of the `FDN` class.

The FDN constructor arguments are:
- `n_M` : number of microphone inputs
- `n_L` : number of loudspeaker outputs
- `fs` : sampling rate
- `nfft` : FFT size
- `alias_decay_db` : anti‑aliasing decay (dB)
- `order` : number of delay lines in the FDN
- `t60_DC` : reverberation time (seconds) at 0 Hz
- `t60_NY` : reverberation time (seconds) at Nyquist frequency

The FDN will automatically generate random feedback matrices, input/output gains, and delay lengths that produce the desired reverberation times.

In [ ]:
n_inputs = 4    # number of microphones
n_outputs = 8   # number of loudspeakers

virtual_room = FDN(
    n_M=n_inputs,
    n_L=n_outputs,
    fs=samplerate,
    nfft=nfft,
    alias_decay_db=alias_decay_db,
    order=16,
    t60_DC=1.0,
    t60_NY=0.2,
)

## 4. Inspect the Virtual Room Object

The `FDN` class is a subclass of a common base class that all virtual rooms share. We can examine its structure and the underlying DSP modules.

In [ ]:
print(f"\nThe {type(virtual_room).__name__} class is a subclass of the {type(virtual_room).__bases__[0].__name__} class.")
print(f"The virtual room was created with {virtual_room.n_M} microphone inputs and {virtual_room.n_L} loudspeaker outputs.")
print(f"Thus, the DSP has {virtual_room.get_v_ML().input_channels} input connections and {virtual_room.get_v_ML().output_channels} output connections.")

print(f"\nThe DSP architecture is a {type(virtual_room.get_v_ML())} instance.")
print("It contains the following modules:")
for module in virtual_room.get_v_ML()._modules.values():
    print(f"  - {type(module).__name__}")

## 5. Process an Impulse Signal

To see how the virtual room transforms a microphone signal, we generate an impulse for each input channel and feed it through the room. The processing is done in the frequency domain:  
1. Apply FFT to the time‑domain impulse.  
2. Pass the spectrum through the DSP (the FDN).  
3. Apply inverse FFT to obtain the output time signal.

We then extract the first channel’s impulse response for plotting.

In [ ]:
# Generate an impulse signal for all input channels
x = signal_gallery(
    batch_size=1,
    n_samples=nfft,
    n=n_inputs,
    signal_type="impulse",
    fs=samplerate
)

# Convert to frequency domain
X = dsp.FFT(nfft=nfft)(x)

# Process through the virtual room
Y = virtual_room.get_v_ML()(X)

# Convert back to time domain
y = dsp.iFFT(nfft=nfft)(Y)

# Extract the first channel for comparison
impulse_in = x.squeeze(0)[:samplerate, 0]          # first second of input channel 0
impulse_out = y.squeeze(0)[:samplerate, 0]        # first second of output channel 0

## 6. Plot the Results

We use the built‑in plotting functions `plot_irs_compare` and `plot_spectrograms_compare` to visualise the impulse response and its spectrogram for the first input/output pair.

In [ ]:
# Compare impulse responses (time domain)
plot_irs_compare(
    ir_1=impulse_in,
    ir_2=impulse_out,
    fs=samplerate,
    label1='Microphone 1',
    label2='Loudspeaker 1'
)

# Compare spectrograms (frequency‑time analysis)
plot_spectrograms_compare(
    ir_1=impulse_in,
    ir_2=impulse_out,
    fs=samplerate,
    nfft=2**11,
    noverlap=2**10,
    label1='Microphone 1',
    label2='Loudspeaker 1'
)

plt.show()

## 7. Conclusion

You have successfully created a virtual room using the `FDN` class, inspected its internal structure, and observed the resulting reverberant impulse response.  

To experiment further:
- Try different `order` or reverberation times.
- Change the number of inputs/outputs.
- Use other virtual room classes (e.g., `unitary_reverberator`, `phase_cancellation`) by replacing the instantiated class.

All classes are documented in `PyRES/virtual_room.py`.